# Лабораторная работа №4. Случайный лес

Чибугаев Илья Алексеевич, вариант 22.

## Данные и метрики

Для классификации взят [Ionosphere](https://archive.ics.uci.edu/dataset/52/ionosphere): 351 радиосигнал и 34 признака. Класс `bad=1` означает сигнал без структуры в ионосфере. Это задача автоматической проверки радиосигналов.

Для регрессии взят [Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength): 1030 составов бетона и 8 признаков. По составу и возрасту нужно оценить прочность в МПа. Такой прогноз полезен перед лабораторными испытаниями смеси.

Лес усредняет деревья, обученные на разных бутстрэп-выборках. Случайный набор признаков в узлах делает деревья менее похожими друг на друга.

Train и test соотносятся как 80/20, `random_state=22`. Классы при разбиении сохраняются в одинаковой пропорции. Подбор параметров идет только на train.

В классификации основной показатель: ROC-AUC. Он не зависит от выбранного порога. F1 показывает качество для `bad`, accuracy дает общую долю верных ответов. В регрессии основной показатель: RMSE в МПа. MAE проще интерпретировать как типичную ошибку, а R² показывает долю объясненной дисперсии.


## Подготовка

Общий код вынесен в `src`, чтобы не повторять его в каждой работе.

In [1]:
# Подключаю общие функции и настраиваю вывод таблиц.
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experiments import TaskExperiment

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Корень проекта: {ROOT}")

Корень проекта: /Users/finleyl/Documents/kimi/tasks/2026-11-10/20-02-04-6933ea61/ml-labs-1-5


## Проверка данных

Смотрю размеры выборок, пропуски и целевые переменные. Эти разбиения используются во всех работах.

In [2]:
# Загружаю данные и проверяю размеры выборок.
experiments = {task: TaskExperiment(4, task) for task in ("classification", "regression")}
for task, experiment in experiments.items():
    data = experiment.data
    print(f"{task}: {data.name}")
    print(data.practical_task)
    print(f"train={data.X_train.shape}, test={data.X_test.shape}, пропуски={int(data.X.isna().sum().sum())}")
    if task == "classification":
        print(f"Доля bad: train={data.y_train.mean():.3f}, test={data.y_test.mean():.3f}")
    else:
        print(data.y.describe().to_string())
    print()

classification: Ionosphere (UCI)
По сигналам фазированной антенной решетки отличать радиосигналы со структурой в ионосфере от проходящих сигналов.
train=(280, 34), test=(71, 34), пропуски=0
Доля bad: train=0.361, test=0.352

regression: Concrete Compressive Strength (UCI)
По составу смеси и возрасту бетона прогнозировать прочность на сжатие в МПа для предварительной оценки рецептуры.
train=(824, 8), test=(206, 8), пропуски=0
count    1030.0000
mean       35.8178
std        16.7057
min         2.3318
25%        23.7071
50%        34.4428
75%        46.1363
max        82.5992



## Бейзлайн sklearn

Беру модель с параметрами по умолчанию. Медианный imputer оставлен как страховка, хотя пропусков в данных нет.

In [3]:
# Обучаю базовые модели sklearn.
for experiment in experiments.values():
    experiment.fit_sklearn_baseline()
baseline_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(baseline_results.drop(columns=["params"]))

classification | sklearn | baseline: accuracy=0.9437, f1=0.9167, roc_auc=0.9948; 0.05 s
regression | sklearn | baseline: rmse=4.5782, mae=3.2030, r2=0.9298; 0.05 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,4,Случайный лес,classification,sklearn,baseline,0.9437,0.9167,0.9948,NaN,NaN,NaN,0.0533
1,4,Случайный лес,regression,sklearn,baseline,NaN,NaN,NaN,4.5782,3.203,0.9298,0.0484


## Подбор параметров

Меняю число деревьев, глубину и размер листа. Для классификации беру `sqrt` признаков в узле, для регрессии 80%. В своей версии оставляю не больше 50 деревьев ради времени расчета.

Использую пятикратную CV. Для классификации выбираю по ROC-AUC, для регрессии по RMSE.

In [4]:
# Подбираю параметры только на обучающей выборке.
for experiment in experiments.values():
    experiment.fit_sklearn_improved()
    print(f"{{experiment.task}}: {{experiment.search.best_params_}}")
    display(experiment.cv_table().head())

classification | sklearn | improved: accuracy=0.9437, f1=0.9167, roc_auc=0.9965; 1.82 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_roc_auc,std_roc_auc,mean_fit_time
0,"{'model__max_depth': None, 'model__min_samples...",0.9743,0.0179,0.1357
1,"{'model__max_depth': None, 'model__min_samples...",0.9734,0.0195,0.0927
2,"{'model__max_depth': 8, 'model__min_samples_le...",0.9714,0.0197,0.1442
3,"{'model__max_depth': 8, 'model__min_samples_le...",0.9700,0.0195,0.0743
4,"{'model__max_depth': None, 'model__min_samples...",0.9664,0.0220,0.1273


regression | sklearn | improved: rmse=4.5147, mae=3.1582, r2=0.9317; 0.61 s
{experiment.task}: {experiment.search.best_params_}


,params,cv_rmse,std_rmse,mean_fit_time
0,"{'model__max_depth': None, 'model__min_samples...",5.3620,0.4477,0.1538
1,"{'model__max_depth': None, 'model__min_samples...",5.3666,0.3967,0.0730
2,"{'model__max_depth': 8, 'model__min_samples_le...",5.6355,0.4473,0.1476
3,"{'model__max_depth': 8, 'model__min_samples_le...",5.6643,0.4344,0.0826
4,"{'model__max_depth': None, 'model__min_samples...",5.7354,0.3879,0.1414


## Что дала настройка

Сравниваю метрики до и после. Для ROC-AUC, F1, accuracy и R² рост означает улучшение, для RMSE и MAE наоборот.

In [5]:
# Считаю разницу метрик между базовой и настроенной моделями.
metric_names = {"classification": ["roc_auc", "f1", "accuracy"], "regression": ["rmse", "mae", "r2"]}
for task, experiment in experiments.items():
    rows = experiment.results.set_index("stage")
    metrics = metric_names[task]
    comparison = rows.loc[["baseline", "improved"], metrics].T
    comparison["change"] = comparison["improved"] - comparison["baseline"]
    print(task)
    display(comparison)

classification


stage,baseline,improved,change
roc_auc,0.9948,0.9965,0.0017
f1,0.9167,0.9167,0.0000
accuracy,0.9437,0.9437,0.0000


regression


stage,baseline,improved,change
rmse,4.5782,4.5147,-0.0636
mae,3.2030,3.1582,-0.0448
r2,0.9298,0.9317,0.0019


## Своя реализация

Лес собран из собственных деревьев из третьей работы. Каждое дерево получает свою выборку и свой seed.

Код лежит в `src/models/`. Сначала обучаю базовую версию на тех же данных, что и sklearn.

In [6]:
# Обучаю базовые версии собственных моделей.
for experiment in experiments.values():
    experiment.fit_custom_baseline()
custom_baseline_results = pd.concat([
    experiment.results[experiment.results["implementation"] == "from_scratch"]
    for experiment in experiments.values()
], ignore_index=True)
display(custom_baseline_results.drop(columns=["params"]))

classification | from_scratch | baseline: accuracy=0.9718, f1=0.9600, roc_auc=0.9983; 0.05 s


regression | from_scratch | baseline: rmse=4.7786, mae=3.2916, r2=0.9235; 1.59 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,4,Случайный лес,classification,from_scratch,baseline,0.9718,0.96,0.9983,NaN,NaN,NaN,0.0510
1,4,Случайный лес,regression,from_scratch,baseline,NaN,NaN,NaN,4.7786,3.2916,0.9235,1.5877


## Своя модель после настройки

Переношу препроцессинг и параметры, выбранные на CV. Ограничение по числу деревьев для своего леса указано выше.

In [7]:
# Применяю найденные параметры к своим моделям.
for experiment in experiments.values():
    experiment.fit_custom_improved()
all_lab_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(all_lab_results.drop(columns=["params"]))

classification | from_scratch | improved: accuracy=0.9437, f1=0.9167, roc_auc=0.9957; 0.09 s


regression | from_scratch | improved: rmse=4.5563, mae=3.2061, r2=0.9305; 2.03 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,4,Случайный лес,classification,sklearn,baseline,0.9437,0.9167,0.9948,NaN,NaN,NaN,0.0533
1,4,Случайный лес,classification,sklearn,improved,0.9437,0.9167,0.9965,NaN,NaN,NaN,1.8164
2,4,Случайный лес,classification,from_scratch,baseline,0.9718,0.9600,0.9983,NaN,NaN,NaN,0.0510
3,4,Случайный лес,classification,from_scratch,improved,0.9437,0.9167,0.9957,NaN,NaN,NaN,0.0903
4,4,Случайный лес,regression,sklearn,baseline,NaN,NaN,NaN,4.5782,3.2030,0.9298,0.0484
5,4,Случайный лес,regression,sklearn,improved,NaN,NaN,NaN,4.5147,3.1582,0.9317,0.6108
6,4,Случайный лес,regression,from_scratch,baseline,NaN,NaN,NaN,4.7786,3.2916,0.9235,1.5877
7,4,Случайный лес,regression,from_scratch,improved,NaN,NaN,NaN,4.5563,3.2061,0.9305,2.0346


## Результаты

Сохраняю все четыре варианта для каждой задачи. Вывод зависит от полученных чисел: если настройка не помогла, это будет видно.

In [8]:
# Сохраняю таблицу и вывожу изменение основной метрики.
output_path = ROOT / "results" / "lab04_results.csv"
output_path.parent.mkdir(exist_ok=True)
all_lab_results.to_csv(output_path, index=False)

def verdict(change, higher_is_better):
    if np.isclose(change, 0):
        return "не изменился"
    improved = change > 0 if higher_is_better else change < 0
    return "улучшился" if improved else "ухудшился"

for task, experiment in experiments.items():
    frame = experiment.results.set_index(["implementation", "stage"])
    metrics = metric_names[task]
    primary = "roc_auc" if task == "classification" else "rmse"
    higher_is_better = primary != "rmse"
    sklearn_base = frame.loc[("sklearn", "baseline"), primary]
    sklearn_best = frame.loc[("sklearn", "improved"), primary]
    custom_base = frame.loc[("from_scratch", "baseline"), primary]
    custom_best = frame.loc[("from_scratch", "improved"), primary]
    print(f"{task}, sklearn: {primary} {sklearn_base:.4f} → {sklearn_best:.4f}, "
          f"{verdict(sklearn_best - sklearn_base, higher_is_better)}.")
    print(f"{task}, своя версия: {primary} {custom_base:.4f} → {custom_best:.4f}, "
          f"{verdict(custom_best - custom_base, higher_is_better)}.")
    print(f"Параметры sklearn: {experiment.search.best_params_}")
print(f"Файл: {output_path.relative_to(ROOT)}")

classification, sklearn: roc_auc 0.9948 → 0.9965, улучшился.
classification, своя версия: roc_auc 0.9983 → 0.9957, ухудшился.
Параметры sklearn: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 150}
regression, sklearn: rmse 4.5782 → 4.5147, улучшился.
regression, своя версия: rmse 4.7786 → 4.5563, улучшился.
Параметры sklearn: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 150}
Файл: results/lab04_results.csv
